# Unificação e Pré-Processamento dos Dados de Estações – QualiAR

Este notebook tem como objetivo **carregar, unificar e preparar** as séries temporais diárias de variáveis meteorológicas e poluentes atmosféricos medidas nas **8 estações de monitoramento do DataRio** na cidade do Rio de Janeiro:

- BANGU  
- CAMPO_GRANDE  
- PEDRA_DE_GUARATIBA  
- IRAJA  
- SAO_CRISTOVAO  
- TIJUCA  
- CENTRO  
- COPACABANA  

## Propósito
1. **Leitura direta** dos arquivos tratados por dia a partir do repositório GitHub do projeto **QualiAR**.  
2. **Padronização** dos nomes de colunas, formatação de datas e inclusão de identificador da estação.  
3. **Concatenação** dos dados de todas as estações em um único `DataFrame`.  
4. (Etapas futuras) Possibilitar:
   - Análise exploratória conjunta da cidade.
   - Agregação diária para cálculo de médias/máximos por município.
   - Geração de estatísticas, gráficos e mapas para avaliação da qualidade do ar.

## Estrutura temporal
Os dados compreendem o período **2012 a 2024** e incluem:
- Variáveis meteorológicas: temperatura, umidade, precipitação (chuva), entre outras.
- Poluentes atmosféricos: CO, NO₂, NOx, SO₂, O₃, PM₁₀, PM₂.₅.
- Índice de Qualidade do Ar (AQI) e classificação qualitativa.

---


## Configurações e importações
Bibliotecas usadas e parâmetros gerais.

In [15]:
import pandas as pd
import glob
import os
import unicodedata
from pathlib import Path
import numpy as np
from pathlib import Path as _Path
import matplotlib.pyplot as plt

OUT_BASE = Path(f"resultados_rio_de_janeiro")
FIG_DIR = OUT_BASE / "figuras"
DATA_DIR = OUT_BASE / "dados"
OUT_BASE.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = _Path(f"resultados_rio_de_janeiro") / "dados"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = _Path(f"resultados_rio_de_janeiro") / "figuras"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
# Helper para criar uma pasta por coluna 
def _col_dir(base_dir, col_name):
    safe = str(col_name).strip().lower().replace(" ", "_").replace("/", "_").replace("\\", "_")
    d = base_dir / safe
    d.mkdir(parents=True, exist_ok=True)
    return d

## Carregamento e Unificação das Estações

Nesta etapa:

1. Lemos os CSVs diários de cada estação diretamente do GitHub 
2. Garantimos que a coluna `data_dia` esteja no formato de data
3. Adicionamos o nome da estação quando não está no arquivo
4. Unimos todos os dados em um único DataFrame
5. Ordenamos por data e salvamos o resultado em `ESTACOES_UNIFICADAS_POR_DIA.csv`

In [17]:
def carregar_estacoes_github(estacoes):
    """
    Lê CSVs diretamente do repositório GitHub (modo raw) e concatena.
    
    estacoes: lista de nomes das estações (strings, ex.: ["BANGU", "CAMPO_GRANDE", ...])
    """
    base_url = "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia"
    
    dfs = []
    for est in estacoes:
        url = f"{base_url}/ESTACAO_{est}_POR_DIA.csv"
        print(f"Lendo: {url}")
        df = pd.read_csv(url, encoding="utf-8")

        if "data_dia" in df.columns:
            df["data_dia"] = pd.to_datetime(df["data_dia"], errors="coerce")
       
        if "nome_estacao" not in df.columns:
            df["nome_estacao"] = est
        
        dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    return df_all

lista_estacoes = ["BANGU", "CAMPO_GRANDE", "PEDRA_DE_GUARATIBA", "IRAJA", "SAO_CRISTOVAO", "TIJUCA", "CENTRO", "COPACABANA"]

df_estacoes = carregar_estacoes_github(lista_estacoes)

df_estacoes.sort_values(by=["data_dia"], inplace=True)

Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_BANGU_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_CAMPO_GRANDE_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_PEDRA_DE_GUARATIBA_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_IRAJA_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_SAO_CRISTOVAO_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_TIJUCA_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_CENTRO_POR_DIA.csv
Lendo: https://raw.githubusercon

In [18]:
project_root = Path().resolve().parents[2]  

output_dir = project_root / "data" / "DataRio" / "Estacoes_Tratadas_Por_Dia"
output_dir.mkdir(parents=True, exist_ok=True)

output_csv_path = output_dir / f"ESTACOES_UNIFICADAS_POR_DIA.csv"
df_estacoes.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"Arquivo salvo em: {output_csv_path}")

Arquivo salvo em: C:\Users\jhter\OneDrive - cefet-rj.br\qualiar\data\DataRio\Estacoes_Tratadas_Por_Dia\ESTACOES_UNIFICADAS_POR_DIA.csv


## Agregação das medições para toda a cidade do Rio de Janeiro

Nesta etapa:

1. Carregamos o arquivo **`ESTACOES_UNIFICADAS_POR_DIA.csv`** contendo as medições diárias de todas as estações.
2. Removemos colunas que não são necessárias para a agregação (`nome_estacao`, `codnum`, `ano`, `mes`, `dia`, `lat`, `lon`, `Qualidade_do_Ar`).
3. Agrupamos os dados pela coluna `data_dia`.
4. Calculamos a **média** de todas as variáveis numéricas para representar os valores diários médios do município.
5. O resultado é um DataFrame (`df_cidade`) com uma linha por dia e colunas contendo as variáveis atmosféricas e poluentes.


In [19]:
cols_to_drop = ["nome_estacao", "codnum", "ano", "mes", "dia", "lat", "lon", "Qualidade_do_Ar"]
df_estacoes = df_estacoes.drop(columns=[c for c in cols_to_drop if c in df_estacoes.columns])

df_estacoes["data_dia"] = pd.to_datetime(df_estacoes["data_dia"], errors="coerce")

df_cidade = df_estacoes.groupby("data_dia").mean(numeric_only=True).reset_index()

display(df_cidade.head())

,data_dia,chuva,temp,ur,co,no,no2,nox,so2,o3,pm10,pm2_5,AQI
0,2012-01-01,12.250,25.834571,92.165000,0.425833,3.613333,23.519667,27.129667,2.673333,23.059286,23.925143,14.619,17.0
1,2012-01-02,56.050,22.836286,95.588571,0.305333,12.675000,27.160333,39.842333,1.793833,20.136429,13.872000,5.083,13.0
2,2012-01-03,0.025,24.947875,76.139250,0.260143,17.175333,28.730667,45.882667,3.918500,15.718375,24.063625,4.208,19.0
3,2012-01-04,0.050,26.006250,72.904125,0.274571,24.745667,40.337667,65.049333,3.123667,25.002500,35.773375,15.729,34.0
4,2012-01-05,0.000,26.498125,75.514500,0.271286,16.643000,34.914000,51.558667,3.066000,33.646250,32.901000,10.917,37.0


In [20]:
for col in df_cidade.columns:
    if col not in ["data_dia", "AQI"]:
        df_cidade[col] = df_cidade[col].round(3)

if "AQI" in df_cidade.columns:
    df_cidade["AQI"] = df_cidade["AQI"].round(0).astype("Int64")

df_cidade["ano"] = df_cidade["data_dia"].dt.year
df_cidade["mes"] = df_cidade["data_dia"].dt.month
df_cidade["dia"] = df_cidade["data_dia"].dt.day

def qual_nivel(idx):
    if pd.isna(idx):
        return np.nan
    if 0 <= idx <= 40:
        return 1
    if 41 <= idx <= 80:
        return 2
    if 81 <= idx <= 120:
        return 3
    if 121 <= idx <= 200:
        return 4
    if 201 <= idx <= 400:
        return 5
    return np.nan

df_cidade["Qualidade_do_Ar"] = df_cidade["AQI"].apply(qual_nivel).astype("Int64")

cols = df_cidade.columns.tolist()

colunas_ordenadas = (
    ["data_dia", "ano", "mes", "dia"] +
    [c for c in cols if c not in ["data_dia", "ano", "mes", "dia"]]
)

df_cidade = df_cidade[colunas_ordenadas]

# Visualizar primeiras linhas
display(df_cidade.head())

,data_dia,ano,mes,dia,chuva,temp,ur,co,no,no2,nox,so2,o3,pm10,pm2_5,AQI,Qualidade_do_Ar
0,2012-01-01,2012,1,1,12.250,25.835,92.165,0.426,3.613,23.520,27.130,2.673,23.059,23.925,14.619,17,1
1,2012-01-02,2012,1,2,56.050,22.836,95.589,0.305,12.675,27.160,39.842,1.794,20.136,13.872,5.083,13,1
2,2012-01-03,2012,1,3,0.025,24.948,76.139,0.260,17.175,28.731,45.883,3.918,15.718,24.064,4.208,19,1
3,2012-01-04,2012,1,4,0.050,26.006,72.904,0.275,24.746,40.338,65.049,3.124,25.002,35.773,15.729,34,1
4,2012-01-05,2012,1,5,0.000,26.498,75.514,0.271,16.643,34.914,51.559,3.066,33.646,32.901,10.917,37,1


In [21]:
project_root = Path().resolve().parents[2]  

output_dir = project_root / "data" / "DataRio" 
output_dir.mkdir(parents=True, exist_ok=True)

output_csv_path = output_dir / f"QUALIAR_RIO_DE_JANEIRO.csv"
df_cidade.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"Arquivo salvo em: {output_csv_path}")

Arquivo salvo em: C:\Users\jhter\OneDrive - cefet-rj.br\qualiar\data\DataRio\QUALIAR_RIO_DE_JANEIRO.csv


## Inspeção inicial
Verificamos dimensões, tipos, amostras, faltantes e duplicados.

In [22]:
print("Dimensões:", df_cidade.shape)
print("\nTipos:")
print(df_cidade.dtypes)

print("\nAmostra:")
display(df_cidade.head())

print("\nValores ausentes por coluna:")
print(df_cidade.isna().sum())

# Duplicados (por carimbo horário e nome_estacao, se houver)
dup_cols = [c for c in ['data_dia'] if c in df_cidade.columns]
if dup_cols:
    ndup = df_cidade.duplicated(subset=dup_cols).sum()
    print(f"\nRegistros duplicados por {dup_cols}: {ndup}")
else:
    print("\nColunas para checar duplicados não disponíveis (nome_estacao/data).")

Dimensões: (4749, 17)

Tipos:
data_dia           datetime64[ns]
ano                         int32
mes                         int32
dia                         int32
chuva                     float64
temp                      float64
ur                        float64
co                        float64
no                        float64
no2                       float64
nox                       float64
so2                       float64
o3                        float64
pm10                      float64
pm2_5                     float64
AQI                         Int64
Qualidade_do_Ar             Int64
dtype: object

Amostra:


,data_dia,ano,mes,dia,chuva,temp,ur,co,no,no2,nox,so2,o3,pm10,pm2_5,AQI,Qualidade_do_Ar
0,2012-01-01,2012,1,1,12.250,25.835,92.165,0.426,3.613,23.520,27.130,2.673,23.059,23.925,14.619,17,1
1,2012-01-02,2012,1,2,56.050,22.836,95.589,0.305,12.675,27.160,39.842,1.794,20.136,13.872,5.083,13,1
2,2012-01-03,2012,1,3,0.025,24.948,76.139,0.260,17.175,28.731,45.883,3.918,15.718,24.064,4.208,19,1
3,2012-01-04,2012,1,4,0.050,26.006,72.904,0.275,24.746,40.338,65.049,3.124,25.002,35.773,15.729,34,1
4,2012-01-05,2012,1,5,0.000,26.498,75.514,0.271,16.643,34.914,51.559,3.066,33.646,32.901,10.917,37,1



Valores ausentes por coluna:
data_dia             0
ano                  0
mes                  0
dia                  0
chuva                0
temp                24
ur                  24
co                  26
no                  24
no2                 24
nox                 24
so2                261
o3                  23
pm10                23
pm2_5              478
AQI                 37
Qualidade_do_Ar     37
dtype: int64

Registros duplicados por ['data_dia']: 0


## Estatísticas descritivas por coluna

In [23]:
desc = df_cidade.describe(include='all').T
desc.to_csv(DATA_DIR / f"estatisticas_rio_de_janeiro.csv")
display(desc)

,count,mean,min,25%,50%,75%,max,std
data_dia,4749,2018-07-02 00:00:00.000000256,2012-01-01 00:00:00,2015-04-02 00:00:00,2018-07-02 00:00:00,2021-10-01 00:00:00,2024-12-31 00:00:00,NaN
ano,4749.0,2018.0,2012.0,2015.0,2018.0,2021.0,2024.0,3.742727
mes,4749.0,6.522215,1.0,4.0,7.0,10.0,12.0,3.449262
dia,4749.0,15.731733,1.0,8.0,16.0,23.0,31.0,8.801904
chuva,4749.0,3.115334,0.0,0.0,0.05,1.875,427.5,12.482548
temp,4725.0,26.008937,15.93,23.601,25.924,28.433,35.825,3.36198
ur,4725.0,71.588067,31.164,64.641,71.643,78.486,98.896,10.732352
co,4723.0,0.360048,0.081,0.269,0.338,0.418,1.475,0.13423
no,4725.0,13.950628,1.643,7.28,11.031,17.012,78.615,10.162804
no2,4725.0,32.660634,6.719,24.425,31.192,38.916,88.312,11.665142


## Séries temporais e médias móveis (7 e 30 dias)
Gera gráficos individuais por variável.

In [24]:
cols_to_analise = [c for c in ['chuva','temp','ur','co','no','no2','nox','so2','o3','pm10','pm2_5', 'AQI'] if c in df_cidade.columns]

In [25]:
def plot_series_with_roll(df_, col, outdir):
    if col not in df_.columns:
        return
    s = df_.set_index('data_dia')[col].sort_index()
    if s.dropna().empty:
        return
    rm7 = s.rolling('7D').mean()
    rm30 = s.rolling('30D').mean()

    plt.figure(figsize=(12,4))
    s.plot(linewidth=0.8, label=col)
    rm7.plot(linewidth=1.0, label='mm7')
    rm30.plot(linewidth=1.2, label='mm30')
    plt.title(f"{col} — Série horária e médias móveis")
    plt.xlabel("Tempo")
    plt.ylabel(col)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir / "serie.png", dpi=150)
    plt.close()

## Distribuições: histograma e boxplot

In [26]:
def plot_distributions(df_, col, outdir):
    if col not in df_.columns:
        return
    x = df_[col].dropna()
    if x.empty:
        return

    # Histograma
    plt.figure(figsize=(6,4))
    plt.hist(x, bins=40)
    plt.title(f"Histograma — {col}")
    plt.xlabel(col); plt.ylabel("Frequência")
    plt.tight_layout()
    plt.savefig(outdir / "hist.png", dpi=150)
    plt.close()

    # Boxplot
    plt.figure(figsize=(4,5))
    plt.boxplot(x.values, vert=True)
    plt.title(f"Boxplot — {col}")
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(outdir / "box.png", dpi=150)
    plt.close()


## Sazonalidade: perfis mensais e anuais

In [27]:
def monthly_profile(df_, col, outdir):
    if col not in df_.columns:
        return
    tmp = df_[['mes', col]].copy()
    g = tmp.groupby('mes')[col].mean(numeric_only=True)
    if g.dropna().empty:
        return
    plt.figure(figsize=(8,3.5))
    plt.plot(g.index, g.values, marker='o')
    plt.title(f"Média mensal — {col}")
    plt.xlabel("Mês"); plt.ylabel(col)
    plt.xticks(range(1,13))
    plt.tight_layout()
    plt.savefig(outdir / "mensal.png", dpi=150)
    plt.close()

def yearly_profile(df_, col, outdir):
    if col not in df_.columns:
        return
    tmp = df_[['ano', col]].copy()
    g = tmp.groupby('ano')[col].mean(numeric_only=True)
    if g.dropna().empty:
        return
    plt.figure(figsize=(8,3.5))
    plt.plot(g.index, g.values, marker='o')
    plt.title(f"Média anual — {col}")
    plt.xlabel("Ano"); plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(outdir / "anual.png", dpi=150)
    plt.close()


## Gerando figuras

In [28]:
for c in cols_to_analise:
    col_dir = _col_dir(FIG_DIR, c)
    plot_series_with_roll(df_cidade, c, col_dir)
    plot_distributions(df_cidade, c, col_dir)
    monthly_profile(df_cidade, c, col_dir)
    yearly_profile(df_cidade, c, col_dir)

print("Figuras salvas por coluna em subpastas de:", FIG_DIR)

Figuras salvas por coluna em subpastas de: resultados_rio_de_janeiro\figuras


## Normalização